In [4]:
!pip install -r ../requirements.txt

In [6]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="너는 튜터야, 튜티들을 환영해줘",
    input="안녕하세요, AI와 첫 대화입니다. 환영해주세요."
)
print(response.output_text)

# response = client.chat.completions.create(
#     model="gpt-4.1-mini",
#     messages=[
#         {"role": "user", "content": "안녕하세요!"}
#     ]
# )
# print(response.choices[0].message.content)

안녕하세요! AI와의 첫 대화를 환영해요. 함께 이야기 나누면서 궁금한 점도 해결하고 재미있게 배워가요. 무엇이든 편하게 물어보세요! 😊


기존에 Open AI와의 대화를 하기 위해서는 completion 방식을 사용했지만 현재 OpenAI는 Response 중심으로 대화하는 것을 권장하고 있다.

Completion 방식에서도 system prompt를 사용하여 답변을 제어 할 수 있듯, Response 방식에서는 instruction 방식으로 제어 할 수 있다.

In [7]:
from pydantic import BaseModel

class ReviewResult(BaseModel):
    sentiment: str
    score: int
    summary: str

client = OpenAI()

response = client.responses.parse(
    model="gpt-4.1-mini",
    input="이 카페는 분위기는 좋은데 커피가 너무 맛 없었어요",
    text_format=ReviewResult
)

print(response.output_parsed)

sentiment='부정적' score=2 summary='카페의 분위기는 좋았으나, 커피 맛이 매우 아쉬웠습니다.'


In [ ]:
LLM의 답변 형식을 강제할 수 있다.

In [ ]:
import json
def get_weather(city):
    return f"{city}의 현재 날씨는 맑음, 18도입니다."

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시의 현재 날씨를 조회한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "날씨를 조회할 도시 이름"
                }
            },
            "required": ["city"],
            "additionalProperties": False
        }
    }
]

response = client.responses.create(
    model="gpt-4.1-mini",
    input="서울 날씨 알려줘.",
    tools=tools
)

print(response.output)


[ResponseFunctionToolCall(arguments='{"city":"서울"}', call_id='call_2vCdUFXr4FYPSBRFBrPMnlQ5', name='get_weather', type='function_call', id='fc_0a80bf56580387590069d444a8d80481a3a9ccc303da4d35e7', namespace=None, status='completed')]


OpenAI Responses API에서 `tools`를 넘기면, 모델은 필요하다고 판단한 경우 직접 답을 만드는 대신 어떤 함수를 호출해야 하는지 요청 할 수 있다.

위 단계에서는 tool을 호출하는 것이 아니라 사용자의 요청에 대해 어떤 tool을 호출하는 것이 맞는지 판단하는 과정이다.

In [17]:
tool_registry = {
    "get_weather": get_weather,
}

tool_outputs = []

for item in response.output:
    if item.type == "function_call":
        tool_name = item.name
        args = json.loads(item.arguments)

        tool_fn = tool_registry[tool_name]
        result = tool_fn(**args)

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result
        })

response2 = client.responses.create(
    model="gpt-4.1-mini",
    previous_response_id=response.id,
    input=tool_outputs
)

print(response2.output_text)

서울의 현재 날씨는 맑고 기온은 18도입니다. 추가 정보나 다른 날씨 관련 질문 있으면 알려주세요!


모델이 `function_call`을 반환하면 사용자에 요청에 대해서 `tool`을 호출해야 한다는 의미이다.
모델이 판단한 function을 실행하고 반환 값을 이전 대화 응답 id와 함께 모델에게 보내면 tool을 사용한 응답을 받을 수 있다.